# Trực quan hóa kết quả

Thay thế ba cell "VISUALIZE" của notebook cũ, vốn viết cho một API chưa từng tồn
tại (`from src.config import cfg`, `cfg.OUT`, `cfg.META_DIM`,
`ClassifierHead(768, ...)`, `student(t, return_attn=True)`) và không chạy được.

Notebook này dùng đúng API hiện hành. Chạy sau khi đã có checkpoint.


In [ ]:
# 0. Bootstrap: lấy code, cài package, nạp cấu hình — chạy được cả Colab lẫn local.
#    Code luôn được clone/pull vào /content (Colab) rồi `pip install -e`, nên không còn
#    phụ thuộc vào việc bạn đã tự chép src/ lên Drive. Chỉ dataset/, artifacts/ và
#    vendor/ nằm trên Drive (bền vững giữa các lần runtime bị ngắt).
import logging, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/hoangtung386/Att-kneenet.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/Challenge_RNSA_KNEE')
    CODE = Path('/content/Att-kneenet')
    if (CODE / '.git').is_dir():
        subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CODE)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[viz,eval]'], check=True)
else:
    CODE = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
    DRIVE_ROOT = CODE
sys.path.insert(0, str(CODE / 'src'))

assert (DRIVE_ROOT / 'dataset' / 'train.csv').is_file(), (
    f'Không thấy dataset tại {DRIVE_ROOT / "dataset"} — hãy đặt dataset RSNA vào đó.'
)
print('code   :', CODE)
print('dataset:', DRIVE_ROOT / 'dataset')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
%matplotlib inline

from knee_mri.constants import LABELS
from knee_mri.data.catalog import StudyCatalog

# 1. Nạp cấu hình
from knee_mri.config import load_config
from knee_mri.utils.logging import setup_logging

RUN = 'pilot'  # 'full' | 'pilot' — phải trùng với MODE ở notebook 02
ARTIFACTS = DRIVE_ROOT / 'artifacts' / ('pilot' if RUN == 'pilot' else '')
cfg = load_config('colab' if IN_COLAB else None, overrides={
    'paths': {
        'data_root': str(DRIVE_ROOT / 'dataset'),
        'artifacts_root': str(ARTIFACTS),
        'dino_code_dir': str(DRIVE_ROOT / 'vendor' / '3DINO'),
    },
})
setup_logging(logging.WARNING)
print('môi trường:', cfg.env, '| dataset:', cfg.paths.data_root, '| artifacts:', cfg.paths.artifacts_root)

## 1. Phân bố xác suất dự đoán

In [ ]:
submission_path = cfg.paths.predictions / 'submission.csv'
if not submission_path.is_file():
    print('Chưa có submission.csv — chạy scripts/predict.py trước.')
else:
    submission = pd.read_csv(submission_path)
    probabilities = submission[list(LABELS)].astype(float).values
    print('Số study:', len(submission))
    print('Xác suất trung bình mỗi nhãn:')
    for name, value in zip(LABELS, probabilities.mean(0)):
        print(f'  {name:<18s} {value:.3f}')

    fig, axes = plt.subplots(3, 4, figsize=(15, 8))
    for ax, name, column in zip(axes.ravel(), LABELS, probabilities.T):
        ax.hist(column, bins=20, color='#4c72b0')
        ax.set_title(name, fontsize=10)
        ax.set_xlim(0, 1)
    fig.suptitle('Phân bố xác suất dự đoán theo nhãn')
    fig.tight_layout()
    plt.show()

## 2. CAM overlay

Kiểm chứng student "nhìn" vào đâu. CAM chỉ dựng được vì
`AttentionBlock` thực sự trả về trọng số attention — bản cũ dùng forward hook
trên `nn.TransformerEncoderLayer`, vốn luôn trả `None`, nên tính năng này chưa
bao giờ chạy.


In [ ]:
from knee_mri.data.normalize import to_unit_range
from knee_mri.inference.predictor import Predictor
from knee_mri.pipeline import compute_cam
from knee_mri.training.checkpoint import BEST_NAME

device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint = cfg.paths.checkpoints / BEST_NAME
catalog = StudyCatalog.from_csv(cfg.paths.train_csv, cfg.paths.train_series_csv)

if not checkpoint.is_file():
    print('Chưa có checkpoint tại', checkpoint, '— chạy notebook 02 trước (và đặt RUN khớp MODE).')
else:
    predictor = Predictor.from_checkpoint(cfg, checkpoint=checkpoint, device=device)

    # Ưu tiên study gold: có nhãn thật để so sánh. compute_cam trả None nếu study không đọc được.
    with_series = set(catalog.studies_with_series())
    candidates = [u for u in catalog.studies_with_gold() if u in with_series] or sorted(with_series)
    explained = study_uid = None
    for study_uid in candidates:
        explained = compute_cam(cfg, predictor, catalog, study_uid)
        if explained is not None:
            break

    if explained is None:
        print('Không dựng được CAM cho study nào.')
    else:
        volume, cam = explained.volume, explained.cam
        print('study:', study_uid[:24])
        print('top-3:', sorted(zip(LABELS, explained.probabilities.round(3)), key=lambda x: -x[1])[:3])
        print('mask ROI:', 'có' if explained.mask is not None else 'chưa sinh (CAM vẫn dựng được, chỉ không đối chiếu IoU)')
        if explained.overlap is not None:
            print('Trùng khớp CAM/ROI:', explained.overlap)

        middle = volume.shape[0] // 2
        display = to_unit_range(volume[middle], cfg.data.norm_mode)
        fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
        axes[0].imshow(display, cmap='gray');                       axes[0].set_title('Lát MRI')
        axes[1].imshow(cam[middle], cmap='inferno');                axes[1].set_title('CAM')
        axes[2].imshow(display, cmap='gray')
        axes[2].imshow(cam[middle], cmap='inferno', alpha=0.45);    axes[2].set_title('Chồng lớp')
        for ax in axes:
            ax.axis('off')
        fig.tight_layout()
        plt.show()

## 3. AUC trên các study có nhãn gold

Đây là tín hiệu duy nhất về metric thật của cuộc thi. 58 study này được giữ làm
validation cố định, không dùng để huấn luyện.


In [ ]:
from knee_mri.inference.predictor import evaluate_gold
from knee_mri.training.metrics import format_auc_report

if not checkpoint.is_file():
    print('Chưa có checkpoint.')
else:
    evaluation = evaluate_gold(predictor, catalog, log_every=0)   # toàn bộ study gold
    if evaluation is None:
        print('Không có study gold nào.')
    else:
        report = evaluation.report
        print('Số study có nhãn gold:', len(evaluation.study_uids))
        print(format_auc_report(report))

        if report.per_label:
            names = list(report.per_label)
            values = [report.per_label[n] for n in names]
            order = np.argsort(values)
            fig, ax = plt.subplots(figsize=(8, 4.5))
            ax.barh([names[i] for i in order], [values[i] for i in order], color='#55a868')
            ax.axvline(0.5, color='crimson', linestyle='--', label='ngẫu nhiên')
            ax.axvline(report.macro, color='navy', linestyle=':', label=f'macro = {report.macro:.3f}')
            ax.set_xlim(0, 1)
            ax.set_xlabel('AUC')
            ax.legend()
            ax.set_title('AUC theo từng nhãn (tập nhãn gold)')
            fig.tight_layout()
            plt.show()